# MLP with Scikit-Learn Pipelines

Companion notebook for Section 8.1 of the MLP lecture notes.

Objectives:

- train an `MLPClassifier` inside a preprocessing pipeline;
- show why feature scaling is essential for MLPs;
- use early stopping and inspect loss curves;
- compare an MLP with a logistic-regression baseline;
- tune MLP hyperparameters with a small grid search.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)


## 1. Dataset

We use the local scikit-learn Breast Cancer dataset. It is small, numeric, and suitable for demonstrating MLP pipelines without network access.


In [ ]:
import warnings
from sklearn.datasets import load_breast_cancer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
target_names = load_breast_cancer().target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Class distribution:', dict(zip(*np.unique(y_train, return_counts=True))))


## 2. Baseline: logistic regression

A linear baseline is useful before training a neural network. It tells us whether the task already has a strong linear solution.


In [ ]:
logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)
logreg_pipe.fit(X_train, y_train)
logreg_pred = logreg_pipe.predict(X_test)

print('Logistic regression accuracy:', accuracy_score(y_test, logreg_pred))
print(classification_report(y_test, logreg_pred, target_names=target_names))


## 3. MLP without scaling versus with scaling

MLPs are sensitive to feature scales. The same architecture behaves much better when the inputs are standardized.


In [ ]:
def fit_with_warning_suppressed(model, X_train, y_train):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        model.fit(X_train, y_train)
    return model

unscaled_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)
fit_with_warning_suppressed(unscaled_mlp, X_train, y_train)

scaled_mlp_pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42,
    ),
)
fit_with_warning_suppressed(scaled_mlp_pipe, X_train, y_train)

comparison = pd.DataFrame([
    {'model': 'MLP without scaling', 'test_accuracy': accuracy_score(y_test, unscaled_mlp.predict(X_test))},
    {'model': 'MLP with StandardScaler', 'test_accuracy': accuracy_score(y_test, scaled_mlp_pipe.predict(X_test))},
])
comparison


## 4. Inspect loss and validation curves

With `early_stopping=True`, scikit-learn keeps a validation score curve in `validation_scores_`.


In [ ]:
scaled_mlp = scaled_mlp_pipe.named_steps['mlpclassifier']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(scaled_mlp.loss_curve_)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')
axes[0].set_title('MLP loss curve')

if hasattr(scaled_mlp, 'validation_scores_'):
    axes[1].plot(scaled_mlp.validation_scores_)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation accuracy')
    axes[1].set_title('Early-stopping validation score')
else:
    axes[1].text(0.5, 0.5, 'No validation_scores_', ha='center')
plt.tight_layout()
plt.show()

print('Epochs run:', scaled_mlp.n_iter_)
print('Best validation score:', getattr(scaled_mlp, 'best_validation_score_', None))


## 5. Final MLP evaluation


In [ ]:
mlp_pred = scaled_mlp_pipe.predict(X_test)
print('MLP accuracy:', accuracy_score(y_test, mlp_pred))
print(classification_report(y_test, mlp_pred, target_names=target_names))

ConfusionMatrixDisplay.from_predictions(y_test, mlp_pred, display_labels=target_names, cmap='Blues')
plt.title('MLP confusion matrix')
plt.show()


## 6. Small hyperparameter search

This grid is intentionally small. The goal is to show the workflow, not exhaustively tune the model.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        activation='relu',
        solver='adam',
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42,
    )),
])

param_grid = {
    'mlp__hidden_layer_sizes': [(32,), (64, 32)],
    'mlp__alpha': [1e-4, 1e-2],
    'mlp__learning_rate_init': [1e-3, 1e-2],
}

grid = GridSearchCV(search_pipe, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    grid.fit(X_train, y_train)

print('Best parameters:', grid.best_params_)
print('Best CV accuracy:', grid.best_score_)


In [ ]:
grid_results = pd.DataFrame(grid.cv_results_)
cols = ['param_mlp__hidden_layer_sizes', 'param_mlp__alpha', 'param_mlp__learning_rate_init', 'mean_test_score', 'std_test_score', 'rank_test_score']
grid_results[cols].sort_values('rank_test_score')


## 7. Test the selected model once

The test set remains separate from the grid search and is used after model selection.


In [ ]:
best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)
print('Best-grid model test accuracy:', accuracy_score(y_test, best_pred))
print(classification_report(y_test, best_pred, target_names=target_names))


## Exercises

1. Add `(128, 64)` to the grid. Does it improve validation performance?
2. Try `activation='tanh'`. Does convergence change?
3. Disable `early_stopping`. How many iterations does the model run, and how does test accuracy change?


## Takeaways

- Always scale numeric features before an MLP.
- A linear baseline helps calibrate expectations.
- Early stopping gives a cheap regularization mechanism.
- Tune architecture, L2 penalty (`alpha`), and learning rate with validation or cross-validation.
